## Description
This notebook demonstrates how to connect to an Azure SQL Database and load the data from the database into Azure Data Lake Storage (ADLS).

In [0]:
from pyspark.sql import functions as F

In [0]:
database = "quant-db-databricks"
server_name = "myquantclound"
jdbc_url = f"jdbc:sqlserver://{server_name}.database.windows.net:1433;database={database};encrypt=true;"
target_base_path = "abfss://adb-container@qcadlsbatch10.dfs.core.windows.net/"

sql_user = "quant_user"
sql_password = dbutils.secrets.get(scope="databricks-secret-scope", key="sql-databricks-passord")
# print(sql_password)

In [0]:
df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("dbtable", "SalesLT.Customer")
    .load()
)

df.display()

In [0]:
info_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("user", sql_user)
    .option("password", sql_password)
    .option("dbtable", "information_schema.tables")
    .load()
)

info_df = (
    info_df
    .filter((F.col("TABLE_TYPE") == "BASE TABLE") & (F.col("TABLE_SCHEMA") != "dbo"))
    .select(F.col("TABLE_SCHEMA"), F.col("TABLE_NAME"))
)
info_df.display()

In [0]:
info_data = info_df.collect()
print(info_data)

In [0]:
for row in info_data:
    table_name = row["TABLE_NAME"]
    table_schema = row["TABLE_SCHEMA"]

    target_table_name = f"{table_schema}_{table_name}"
    df = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("user", sql_user)
        .option("password", sql_password)
        .option("dbtable", f"{table_schema}.{table_name}")
        .load()
    )

    (
        df.write.mode("overwrite")
        .option("path", f"{target_base_path}/sql_data/{target_table_name.lower()}")
        .saveAsTable(target_table_name.lower())
    )